# Decision Tree: Train and Compare

**Motivation:** logistic regression gives every feature one linear weight. The EDA showed patterns a linear model cannot follow:

- **Bends:** `temp_min` drops a lot from Good to Poor, then goes flat.
- **U-shapes:** `humidity_mean` falls, then rises again.
- **Combinations:** Severe days often need two things at once, *high AQI **and** stagnant air*.

A decision tree chains yes/no threshold splits (in the trained tree: `aqi_today` above 204, then the 7-day average above 178), so it can follow bends and combinations.

**Scope:** train a decision tree, tune its size, and compare it with **persistence** (the bar, macro F1 0.651) and the best **logistic regression** (`models/lr_best.bin`) on the same 2023 days, using macro F1, precision, recall, Severe misses and false alarms, and the Dangerous alert.

Decisions so far: [`docs/model_decisions.md`](../docs/model_decisions.md).

## 1. Setup

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree

import sys
sys.path.insert(0, '../scripts')                 # utility.py lives in scripts/
from utility import (AQI_CATEGORIES, prepare_dataset, split_train_val_test, add_lr_features,
                     forecast_scorecard, class_scores_table)

DATA_PATH = '../data/processed/aqi_weather_2015_2025.csv'
MODEL_DIR = Path('models')
TARGET = 'tomorrow_status'
TREND = ['aqi_roll7_mean', 'aqi_roll7_std', 'aqi_diff1', 'aqi_dev7']
pd.set_option('display.max_columns', 30)

## 2. The same days as every other model

The same data pipeline, the same 195 rows dropped and the same time split, so every model is scored on exactly the same 2023 days.

In [ ]:
df = prepare_dataset(DATA_PATH)
before = len(df)
df = df[df[TREND].notna().all(axis=1)]                   # the same 195 rows every model drops
df_train, df_val, df_test = split_train_val_test(df)

print(f'Dropped {before - len(df)} rows with empty trend features')
print('train:', len(df_train), '| validation:', len(df_val), '| test:', len(df_test))

## 3. Features: raw columns, no transforms

Trees split on thresholds, so they do **not** need the transforms logistic regression needed:

| Logistic regression needed | Tree uses instead | Why the tree does not care |
|---|---|---|
| `pressure_rel` | `pressure_mean` | A tree can split per city on its own |
| `precip_log` | `precip_sum` | A threshold ("rain above 2 mm") works the same on any scale |
| `month_sin`, `month_cos` | `month` | A tree can ask "month ≥ 11 or ≤ 1" directly |
| Scaling | none | Thresholds do not depend on the scale |
| Dropping overlapping features (VIF) | keeps all of them | Overlapping features do not make a tree unstable |

So the tree gets all raw weather columns and **all four** trend features, including `aqi_roll7_mean`. `DictVectorizer` still turns `city` into yes/no columns, and keeps the dictionary input format of the web service.

In [ ]:
TREE_NUMERIC = (['aqi_today'] + TREND +
                ['temp_mean', 'temp_max', 'temp_min', 'temp_range', 'humidity_mean', 'dew_point_mean',
                 'precip_sum', 'pressure_mean', 'cloud_cover_mean', 'wind_speed_mean', 'wind_speed_max',
                 'wind_u_mean', 'wind_v_mean', 'month'])
TREE_BINARY = [c for c in df.columns if c.startswith('has_')] + ['aqi_imputed']
TREE_CATEGORICAL = ['city']
FEATURES = TREE_NUMERIC + TREE_BINARY + TREE_CATEGORICAL

def to_records(part):
    return part[FEATURES].to_dict(orient='records')

X_train, X_val = to_records(df_train), to_records(df_val)
y_train = df_train[TARGET].astype(str).to_numpy()
y_val = df_val[TARGET].astype(str).to_numpy()

assert df_train[FEATURES].notna().all().all() and df_val[FEATURES].notna().all().all(), 'missing values in the features'
print(f'{len(TREE_NUMERIC)} numeric + {len(TREE_BINARY)} yes/no + {len(TREE_CATEGORICAL)} text = {len(FEATURES)} features')

In [ ]:
def make_tree(max_depth=None, min_samples_leaf=1, class_weight=None):
    return Pipeline([
        ('dv', DictVectorizer(sparse=False)),             # city -> one-hot; numbers pass through
        ('tree', DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=min_samples_leaf,
                                        class_weight=class_weight, random_state=1)),
    ])

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=AQI_CATEGORIES, average='macro')

## 4. Unconstrained tree: overfitting

A tree with no size limits.

In [ ]:
free_tree = make_tree().fit(X_train, y_train)            # no limits at all

print('Leaves:', free_tree['tree'].get_n_leaves(), '| depth:', free_tree['tree'].get_depth())
print(f'Macro F1 on training years: {macro_f1(y_train, free_tree.predict(X_train)):.3f}')
print(f'Macro F1 on 2023:           {macro_f1(y_val, free_tree.predict(X_val)):.3f}   (persistence 0.651)')

**This is overfitting.** The tree grew **6,700 leaves**, about one leaf per 5 training days. It memorised the training years (macro F1 **1.000**) but scores only **0.513** on 2023, far below persistence.

Logistic regression shows no gap (training 0.649, 2023 0.647): it is too simple to memorise. A tree must be constrained; two settings do that:

| Setting | Meaning | Effect |
|---|---|---|
| `max_depth` | How many questions in a row the tree may ask | Fewer questions → simpler tree |
| `min_samples_leaf` | How many training days every final group (leaf) must contain | A leaf of 100 days cannot memorise one odd day |

## 5. Tuning tree size

Grid: 6 depths × 5 leaf sizes × 2 class weightings = 60 trees, each scored on the training years and on 2023.

In [ ]:
DEPTHS = [3, 5, 8, 10, 12, None]                         # None = no depth limit
LEAVES = [1, 50, 100, 200, 400]                          # minimum days in every leaf
WEIGHTS = {'no class weights': None, 'balanced': 'balanced'}

rows, fitted = [], {}
for weight_name, class_weight in WEIGHTS.items():
    for depth in DEPTHS:
        for leaf in LEAVES:
            m = make_tree(depth, leaf, class_weight).fit(X_train, y_train)
            pred_tr, pred_va = m.predict(X_train), m.predict(X_val)
            rows.append({
                'weights': weight_name, 'max_depth': depth, 'min_samples_leaf': leaf,
                'leaves': m['tree'].get_n_leaves(),
                'train_macro_f1': macro_f1(y_train, pred_tr),
                'val_macro_f1': macro_f1(y_val, pred_va),
                'said_severe': (pred_va == 'Severe').sum(),
            })
            fitted[(weight_name, depth, leaf)] = m

tuning = pd.DataFrame(rows)
tuning['gap'] = tuning['train_macro_f1'] - tuning['val_macro_f1']
print(len(tuning), 'trees trained')
tuning.sort_values('val_macro_f1', ascending=False).head(10).round(3)

**Macro F1 on 2023** for every combination (rows: weighting and depth, columns: minimum days per leaf; NaN depth = no limit):

In [ ]:
(tuning.pivot_table(index=['weights', 'max_depth'], columns='min_samples_leaf', values='val_macro_f1', dropna=False)
       .reindex(['no class weights', 'balanced'], level=0)
       .round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
part = tuning[tuning['weights'] == 'no class weights'].copy()
part['depth'] = part['max_depth'].fillna(20)             # plot "no limit" at the right end
for ax, leaf in zip(axes, [1, 100]):
    p = part[part['min_samples_leaf'] == leaf]
    ax.plot(p['depth'], p['train_macro_f1'], marker='o', label='training years')
    ax.plot(p['depth'], p['val_macro_f1'], marker='o', label='validation (2023)')
    ax.axhline(0.651, color='grey', linestyle='--', linewidth=1, label='persistence (0.651)')
    ax.set_title(f'min_samples_leaf = {leaf}')
    ax.set_xticks([3, 5, 8, 10, 12, 20], ['3', '5', '8', '10', '12', 'no limit'])
    ax.set_xlabel('max_depth')
axes[0].set_ylabel('macro F1')
axes[0].legend()
plt.tight_layout()
plt.show()

**Findings:**
- **Overfitting is controlled by the leaf size.** With `min_samples_leaf = 1` (left plot), the training score climbs to 1.0 as the tree deepens, while the 2023 score falls (0.643 → 0.513). With `min_samples_leaf = 100` (right plot), the two lines stay together (gap about 0.01).
- **Once leaves need 100 days, depth stops mattering.** Depth 10, 12 and no limit all give 0.658: the tree cannot split further anyway.
- **Balanced class weights hurt again** (best 0.636), for the same reason as in logistic regression: they lower the bar for the rare categories.
- **The scores jump between neighbouring settings** (0.658 at 100 days per leaf, 0.650 at 200, 0.648 at 400). A single tree is sensitive to small changes, so differences of about 0.01 are partly luck.

## 6. The best tree

Highest 2023 macro F1; if several are within 0.002, the one with the fewest leaves (simplest).

In [ ]:
best_score = tuning['val_macro_f1'].max()
best = (tuning[tuning['val_macro_f1'] >= best_score - 0.002]
        .sort_values('leaves')                            # within 0.002 of the best: the simplest tree
        .iloc[0])
best_key = (best['weights'], None if pd.isna(best['max_depth']) else int(best['max_depth']), int(best['min_samples_leaf']))
best_tree = fitted[best_key]
pred_tree = best_tree.predict(X_val)

print(f"Best tree: {best['weights']}, max_depth={best_key[1]}, min_samples_leaf={best_key[2]}, "
      f"{best['leaves']} leaves -> 2023 macro F1 {best['val_macro_f1']:.3f} (training {best['train_macro_f1']:.3f})")

## 7. Compare: persistence vs logistic regression vs decision tree

Logistic regression is loaded from `models/lr_best.bin`. It needs its own features (`pressure_rel`, `precip_log`, month on a circle), which are rebuilt with the city pressure normals saved in its bundle.

In [ ]:
with open(MODEL_DIR / 'lr_best.bin', 'rb') as f:
    lr_bundle = pickle.load(f)

df_val_lr = add_lr_features(df_val, pd.Series(lr_bundle['city_pressure']))   # the LR needs its own transforms
lr_features = sum(lr_bundle['features'].values(), [])
pred_lr = lr_bundle['model'].predict(df_val_lr[lr_features].to_dict(orient='records'))

print(f"Logistic regression (class_weight={lr_bundle['class_weight']}, C={lr_bundle['C']}): "
      f"2023 macro F1 {macro_f1(y_val, pred_lr):.3f}")

**The scorecard** (`forecast_scorecard` in `utility.py`, shared by every model notebook):

In [ ]:
PREDICTIONS = {
    'persistence': df_val['status_today'].astype(str).to_numpy(),
    'logistic regression': pred_lr,
    'decision tree': pred_tree,
}
scorecard = forecast_scorecard(y_val, PREDICTIONS, y_today=df_val['status_today'])
scorecard

**Precision, recall and F1 for every category** (`class_scores_table` in `utility.py`):

In [ ]:
class_scores_table(y_val, PREDICTIONS).round(2)

**Confusion matrix of the tree** (rows: what really happened, columns: what the tree forecast):

In [ ]:
pd.DataFrame(confusion_matrix(y_val, pred_tree, labels=AQI_CATEGORIES),
             index=pd.Index(AQI_CATEGORIES, name='actual'),
             columns=pd.Index(AQI_CATEGORIES, name='predicted'))

**Findings:**

| 2023 | Persistence | Logistic regression | Decision tree |
|---|---|---|---|
| Macro F1 | 0.651 | 0.647 | **0.658** |
| Accuracy | 0.710 | **0.729** | 0.723 |
| Severe caught / missed / false alarms | 27 / 24 / 25 | 20 / 31 / **11** | 25 / 26 / 19 |
| Dangerous recall / precision | 0.749 / 0.753 | **0.807 / 0.761** | 0.768 / 0.740 |
| Change days right | 0% | **35.3%** | 27.8% |
| Unchanged days right | **100%** | 88.3% | 90.4% |

- **The tree is the first model to beat persistence on macro F1** (0.658 against 0.651), but the margin (0.007) is smaller than the jumps between neighbouring tree settings (about 0.01): a **narrow, fragile** win. Section 9 checks it on five years.
- **Tree better than logistic regression at the rare ends:** Good F1 0.62 against 0.52 (156 of 293 Good days caught, logistic regression 115), Severe F1 0.53 against 0.49 (25 Severe days caught instead of 20). This is what lifts its macro F1.
- **Logistic regression better on the middle categories and the Dangerous alert:** higher recall (81% against 77%) *and* precision (76% against 74%). It also gets more change days right (35% against 28%).
- **The tree stays closer to persistence:** more unchanged days right (90%), fewer change days. It is the more cautious model.

## 8. What did the tree learn?

**Feature importance:** how much each feature reduced the tree's mistakes, as a share of the total.

In [ ]:
importance = pd.Series(best_tree['tree'].feature_importances_, index=best_tree['dv'].feature_names_)
importance.sort_values(ascending=False).head(12).round(3).to_frame('importance')

**The first three levels of the tree.** Each box shows the question, the share of training days that reach it, and the mix of categories in it.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(best_tree['tree'], max_depth=2, feature_names=best_tree['dv'].feature_names_,
          class_names=list(best_tree['tree'].classes_), filled=True, impurity=False,
          proportion=True, fontsize=9, ax=ax)
ax.set_title('Top 3 levels of the best tree (each box: the question, share of days, class mix)')
plt.show()

**Findings:**
- **`aqi_today` does 83% of the work.** The tree first sorts days by today's AQI, like persistence, then refines with a few other features.
- **The tree recovered the official category limits from the data.** Its first questions are about `aqi_today` at 49.5, 103.5, 204.5 and 307.5, next to the CPCB boundaries at 50 (Good / Satisfactory), 100 (Satisfactory / Moderate), 200 (Moderate / Poor) and 300 (Poor / Very Poor).
- Next most useful: the **weekly average** (`aqi_roll7_mean`), **rain** and **maximum wind speed**, the weather that clears pollution.
- The tree is largely "persistence with corrections", which explains why it scores close to persistence and why the win is narrow.

## 9. Rolling validation: checking the 2023 win

All results above use **one** year (2023), which is noisy: tree scores moved by about 0.01 between neighbouring settings, more than the tree's margin over persistence.

**Rolling validation** repeats the comparison on five years. Each round trains only on the years **before** the year it scores, as in real use:

| Round | Learns from | Scores on |
|---|---|---|
| 1 | 2015–2018 | 2019 |
| 2 | 2015–2019 | 2020 |
| 3 | 2015–2020 | 2021 |
| 4 | 2015–2021 | 2022 |
| 5 | 2015–2022 | 2023 |

`rolling_validation` in `utility.py` refuses to score 2024 or later, so the test years stay locked. Anything learned from data (the logistic regression's city pressure normals) is recomputed in each round from that round's training years only.

The chosen tree is compared with its neighbouring settings and with logistic regression (same settings as `lr_best.bin`).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from utility import rolling_validation

df_dev = pd.concat([df_train, df_val])            # 2015-2023 only; the test years stay locked

def tree_xy(train, val):
    return to_records(train), to_records(val)

LR_FEATURES = sum(lr_bundle['features'].values(), [])

def lr_xy(train, val):
    city_pressure = train.groupby('city')['pressure_mean'].mean()     # from this round's training years only
    train_lr, val_lr = add_lr_features(train, city_pressure), add_lr_features(val, city_pressure)
    return train_lr[LR_FEATURES].to_dict(orient='records'), val_lr[LR_FEATURES].to_dict(orient='records')

def make_lr():
    return Pipeline([
        ('dv', DictVectorizer(sparse=False)),
        ('scale', StandardScaler()),
        ('lr', LogisticRegression(class_weight=lr_bundle['class_weight'], C=lr_bundle['C'],
                                  max_iter=5000, random_state=1)),
    ])

In [ ]:
CANDIDATES = {                                    # name: (make a fresh model, build its features)
    'tree: depth 10, leaf 100 (chosen)': (lambda: make_tree(10, 100), tree_xy),
    'tree: depth 10, leaf 50':           (lambda: make_tree(10, 50), tree_xy),
    'tree: depth 10, leaf 200':          (lambda: make_tree(10, 200), tree_xy),
    'tree: depth 10, leaf 400':          (lambda: make_tree(10, 400), tree_xy),
    'tree: depth 5, leaf 1':             (lambda: make_tree(5, 1), tree_xy),
    'logistic regression':               (make_lr, lr_xy),
}

rolling = pd.concat({name: rolling_validation(df_dev, make, xy) for name, (make, xy) in CANDIDATES.items()},
                    names=['model']).reset_index(level=0)
rolling.round(3)

**Macro F1 per year** (persistence first, for reference):

In [ ]:
by_year = rolling.pivot_table(index='model', columns='val_year', values='macro_f1')
persistence_row = rolling.groupby('val_year')['persistence_macro_f1'].first().rename('persistence')
by_year = pd.concat([by_year, persistence_row.to_frame().T]).reindex(['persistence'] + list(CANDIDATES))
by_year.round(3)

**Summary over the five years** (`spread` = how much the score moves from year to year; `mean_train_gap` = training score minus validation score):

In [ ]:
summary = rolling.groupby('model', sort=False).agg(
    mean_macro_f1=('macro_f1', 'mean'),
    spread=('macro_f1', 'std'),                              # how much the score moves from year to year
    mean_vs_persistence=('vs_persistence', 'mean'),
    years_beating_persistence=('vs_persistence', lambda d: f'{(d > 0).sum()} of {len(d)}'),
    mean_train_gap=('train_macro_f1', 'mean'),
)
summary['mean_train_gap'] = summary['mean_train_gap'] - summary['mean_macro_f1']
summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for name in ['persistence', 'logistic regression', 'tree: depth 10, leaf 100 (chosen)', 'tree: depth 10, leaf 400']:
    style = dict(color='grey', linestyle='--') if name == 'persistence' else {}
    ax.plot(by_year.columns, by_year.loc[name], marker='o', label=name, **style)
ax.set_xticks(list(by_year.columns))
ax.set_xlabel('validation year (learned from all earlier years)')
ax.set_ylabel('macro F1')
ax.set_title('Rolling validation: the same models, five different years')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Findings:**
- **Over five years the tree ties with persistence.** The chosen tree averages **0.631**, persistence 0.634: a mean gap of **−0.003**. It wins in 3 of the 5 years by a hair (+0.002 to +0.006) and loses clearly in 2022 (−0.023). The +0.007 on 2023 was a good year, not a reliable win.
- **Logistic regression is worse over five years than 2023 suggested:** mean 0.613, **−0.021** below persistence, ahead only in 2021. It loses badly in 2019 (−0.065, learned from only four years) and in 2022 (−0.045). Its scores also vary most from year to year (spread 0.051).
- **The chosen tree setting holds up.** At least 100 days per leaf is still among the best (mean 0.631, tied with 200). Smaller leaves (50) overfit (training gap 0.050) and never beat persistence; larger leaves (400) or a shallow tree (depth 5) are too simple and never beat it either.
- **Years differ more than models do.** Persistence itself ranges from 0.609 (2019) to 0.655 (2020), so one validation year can make a model look better or worse than it is.
- **2022 is hard for every learned model.** It had only 32 Severe days, far fewer than the training years (the drift seen earlier), so models trained on more polluted years expect more bad days than 2022 delivered.

**Implication:** no model so far **reliably** beats "tomorrow = today". From now on, a model only counts as better if it beats persistence **on average over the five years and in most of them**, not just in 2023.

## 10. Save the model

In [ ]:
MODEL_PATH = MODEL_DIR / 'dt_best.bin'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
bundle = {
    'model': best_tree,                                    # DictVectorizer + decision tree
    'params': {'max_depth': best_key[1], 'min_samples_leaf': best_key[2], 'class_weight': WEIGHTS[best_key[0]]},
    'features': {'numeric': TREE_NUMERIC, 'binary': TREE_BINARY, 'categorical': TREE_CATEGORICAL},
    'categories': AQI_CATEGORIES,
    'cities': sorted(df_train['city'].unique()),           # the service must reject any other city
    'validation': scorecard['decision tree'].to_dict(),
    'tuning': tuning.round(4).to_dict(orient='records'),
}
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(bundle, f)
with open(MODEL_PATH, 'rb') as f:
    assert (pickle.load(f)['model'].predict(X_val) == pred_tree).all()
print(f'Saved and verified: {MODEL_PATH.resolve()}')

## Discussion

1. **A tree must be constrained, or it memorises.** Without limits: 1.000 on training, 0.513 on 2023. With at least 100 days per leaf: 0.668 and 0.658. Logistic regression, too simple to memorise, never showed this overfitting.
2. **On 2023 the tree narrowly beats persistence** (0.658 vs 0.651), mostly by doing better than logistic regression at the **rare ends** (Good and Severe). **Over five years it only ties** (mean −0.003, ahead in 3 of 5 years), and logistic regression falls behind (mean −0.021).
3. **No model wins everything yet:**
   - for the six-category score (macro F1): **decision tree**;
   - for days right and the **Dangerous** health alert: **logistic regression**;
   - for Severe false alarms: **logistic regression** (11);
   - for Severe days caught: **persistence** (27), the tree close behind (25).
4. **One validation year is not enough.** A single tree's score moves by about 0.01 between neighbouring settings, and persistence itself varies from 0.609 to 0.655 between years. Rolling validation over 2019–2023 is now the standard check for every model.

**Next: random forest** (many trees on slightly different samples of the data, averaged, which removes much of a single tree's variance), then XGBoost. Both are judged with the same `forecast_scorecard` on 2023 **and** with `rolling_validation`; the target is to beat persistence on average over the five years and in most of them.